In [1]:
# %% [markdown]
# # 00_fetch_phase1_data.ipynb
# **SCRIPT VERSION: 2026-07-15-v3** (check this line printed at the top
# of cell output below matches, to confirm you're running the latest
# file and not a stale copy from an earlier paste/cell)
#
# Fetches Phase 1 (ACME Insurance) benchmark data from the dbt-labs
# repository. This is the pre-computed experiment data referenced
# in the manuscript (Section 3.3.1): 11 questions x 24 LLM models x
# 20 iterations = 5,170 records.
#
# Source: https://github.com/dbt-labs/dbt-llm-sl-bench
# The data lives in `llm_bench_dashboard.db`, a DuckDB file (not
# SQLite -- requires the `duckdb` Python package to read).
#
# IMPORTANT PROVENANCE NOTE: this repository is actively maintained
# and its `main` branch has been updated since the manuscript's
# original Phase 1 re-analysis was performed. The model list found
# in a fresh clone includes models released well after the paper's
# stated experiment period (e.g. gpt-5.x, claude-opus-4-6 generation
# models), which means `main` keeps growing over time. If the original
# analysis was pinned to a specific commit or release, that should be
# used instead of a fresh `main` clone to guarantee reproducibility.
# This notebook records the exact commit hash it fetched (see the
# last cell) specifically so this can be checked and cited in the
# manuscript's data availability statement.

# %%
print("SCRIPT VERSION: 2026-07-15-v3")

import subprocess
import os
import shutil

# %% [markdown]
# ## 1. Clone (or update) the dbt-labs benchmark repository

# %%
REPO_URL = "https://github.com/dbt-labs/dbt-llm-sl-bench.git"
REPO_DIR = "./dbt-llm-sl-bench"

if os.path.exists(REPO_DIR):
    print(f"'{REPO_DIR}' already exists.")
    resp = input("Delete and re-clone fresh? [y/N]: ").strip().lower()
    if resp == 'y':
        shutil.rmtree(REPO_DIR)
    else:
        print("Using existing clone as-is (skipping re-clone).")

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("CLONE FAILED:")
        print(result.stderr)
        raise RuntimeError(
            "git clone failed. Check that git is installed and this "
            "machine has network access to github.com."
        )
    print("Clone successful.")

# %% [markdown]
# ## 2. Record the exact commit hash fetched (for reproducibility /
#     data availability statement)

# %%
commit_result = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
    capture_output=True, text=True
)
if commit_result.returncode != 0:
    print("!! git rev-parse FAILED:")
    print(f"   returncode: {commit_result.returncode}")
    print(f"   stderr: {commit_result.stderr}")
    print(f"   Attempted to run in REPO_DIR = '{os.path.abspath(REPO_DIR)}'")
    print(f"   Does that directory exist? {os.path.isdir(REPO_DIR)}")
    if os.path.isdir(REPO_DIR):
        print(f"   Does it contain a .git folder? {os.path.isdir(os.path.join(REPO_DIR, '.git'))}")
    raise RuntimeError(
        "Could not determine the cloned commit hash -- see diagnostic "
        "output above. Common cause: REPO_DIR exists but is not actually "
        "a git repository (e.g. it was manually created or extracted "
        "from a zip rather than cloned). Delete the folder and re-run "
        "this notebook from the top to get a fresh git clone."
    )
commit_hash = commit_result.stdout.strip()

commit_date_result = subprocess.run(
    ["git", "-C", REPO_DIR, "log", "-1", "--format=%cI"],
    capture_output=True, text=True
)
if commit_date_result.returncode != 0:
    print("!! git log FAILED:")
    print(f"   stderr: {commit_date_result.stderr}")
    commit_date = "(unknown -- git log failed)"
else:
    commit_date = commit_date_result.stdout.strip()

print(f"Cloned commit: {commit_hash}")
print(f"Commit date  : {commit_date}")

if not commit_hash or commit_hash == "HEAD":
    print()
    print("!! WARNING: commit_hash looks wrong (empty or literally 'HEAD').")
    print("   This should never happen if git rev-parse succeeded above --")
    print("   if you're seeing this, you are likely running a STALE/CACHED")
    print("   version of this cell rather than the current file content.")
    print("   In Jupyter: re-paste this entire script's latest content into")
    print("   a fresh cell (or restart the kernel and re-run from the top)")
    print("   rather than re-running a previously-pasted older cell.")

print()
print("RECORD THIS in the manuscript's data availability statement or")
print("methods section, e.g.:")
print(f'  "Phase 1 data was fetched from the dbt-labs/dbt-llm-sl-bench')
print(f'  repository at commit {commit_hash[:10]} ({commit_date[:10]})."')

from datetime import datetime, timezone
with open("./phase1_data_provenance.txt", "w") as f:
    f.write(f"repo: {REPO_URL}\n")
    f.write(f"commit: {commit_hash}\n")
    f.write(f"commit_date: {commit_date}\n")
    f.write(f"fetched_at: {datetime.now(timezone.utc).isoformat()}\n")
print("\nSaved provenance record to ./phase1_data_provenance.txt")

# %% [markdown]
# ## 3. Open the DuckDB source file and inspect contents

# %%
import duckdb

DUCKDB_PATH = os.path.join(REPO_DIR, "llm_bench_dashboard.db")

if not os.path.exists(DUCKDB_PATH):
    raise FileNotFoundError(
        f"Expected DuckDB file not found at {DUCKDB_PATH}. The "
        f"repository structure may have changed -- inspect {REPO_DIR} "
        f"manually to locate the results database."
    )

conn = duckdb.connect(DUCKDB_PATH, read_only=True)

tables = conn.execute("SELECT table_name FROM information_schema.tables").fetchdf()
print("Tables found:")
for t in tables['table_name']:
    count = conn.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
    print(f"  {t}: {count} rows")

# %% [markdown]
# ## 4. Verify record counts match the manuscript's stated numbers
#     (Section 3.3.1: "11 questions across 24 LLM models and 20
#     iterations (n = 5,170 records)")

# %%
n_questions = conn.execute("SELECT COUNT(*) FROM challenge_text_info").fetchone()[0]
n_total_records = conn.execute("SELECT COUNT(*) FROM sql_answers").fetchone()[0]
n_models = conn.execute("SELECT COUNT(DISTINCT model) FROM sql_answers").fetchone()[0]
method_counts = conn.execute(
    "SELECT method, COUNT(*) as n FROM sql_answers GROUP BY method"
).fetchdf()

print(f"Questions (challenge_text_info)      : {n_questions}  (manuscript states: 11)")
print(f"Total records (sql_answers)          : {n_total_records}  (manuscript states: 5,170)")
print(f"Distinct models                      : {n_models}  (manuscript states: 24)")
print(f"\nRecords per method:")
print(method_counts)

print()
if n_questions == 11 and n_total_records == 5170:
    print("✓ Question count and total record count MATCH the manuscript.")
else:
    print("✗ MISMATCH with manuscript's stated numbers -- investigate before")
    print("  using this data for re-analysis. Possible causes: repo has been")
    print("  updated since the original analysis (see provenance note above),")
    print("  or the original analysis filtered/subset this data in a way not")
    print("  fully documented in the manuscript.")

if n_models != 24:
    print(f"\n! Model count is {n_models}, manuscript states 24 -- this is the")
    print("  clearest signal of whether the repo has grown since the original")
    print("  analysis (more models added over time). List the actual models")
    print("  below and compare against what would be expected as of the")
    print("  manuscript's writing.")

print("\nDistinct models in this data:")
models_df = conn.execute("SELECT DISTINCT model FROM sql_answers ORDER BY model").fetchdf()
for m in models_df['model']:
    print(f"  {m}")

# %% [markdown]
# ## 5. Export to CSV for downstream analysis
#     (keeps phase1_analysis.ipynb decoupled from DuckDB/git specifics --
#     it can just read a CSV like the other phases do)

# %%
df_answers = conn.execute("SELECT * FROM sql_answers").fetchdf()
df_questions = conn.execute("SELECT * FROM challenge_text_info").fetchdf()

df_answers.to_csv("./phase1_sql_answers_raw.csv", index=False, encoding='utf-8-sig')
df_questions.to_csv("./phase1_questions_raw.csv", index=False, encoding='utf-8-sig')

print(f"Exported:")
print(f"  ./phase1_sql_answers_raw.csv  ({len(df_answers)} rows)")
print(f"  ./phase1_questions_raw.csv    ({len(df_questions)} rows)")

conn.close()

# %% [markdown]
# ## 6. Quick sanity check: reproduce the manuscript's headline
#     Phase 1 accuracy figures (SL: 75.4%, SQL: 62.9%) as a smoke test

# %%
import pandas as pd

df = pd.read_csv("./phase1_sql_answers_raw.csv")

acc_by_method = df.groupby('method')['is_correct'].mean()
print("Overall accuracy by method (this fetched data):")
print(acc_by_method)
print()
print("Manuscript states: SL=75.4%, SQL=62.9%")
print()

sl_acc = acc_by_method.get('semantic_layer', float('nan'))
sql_acc = acc_by_method.get('sql', float('nan'))
print(f"This data: SL={sl_acc:.1%}, SQL={sql_acc:.1%}")

if abs(sl_acc - 0.754) < 0.01 and abs(sql_acc - 0.629) < 0.01:
    print("\n✓ Matches manuscript's reported figures closely.")
else:
    print("\n! Does NOT closely match manuscript's reported figures.")
    print("  This could mean the repo has accumulated additional model")
    print("  results since the original analysis, changing the aggregate.")
    print("  Consider filtering to only the models available as of the")
    print("  manuscript's original analysis date, if that list is known,")
    print("  or explicitly noting the discrepancy and updated figures in")
    print("  the revision.")

SCRIPT VERSION: 2026-07-15-v3
Cloning https://github.com/dbt-labs/dbt-llm-sl-bench.git ...
Clone successful.
Cloned commit: c377468c59ed4b6a5739d87e929cdebc77ba2d32
Commit date  : 2026-04-07T18:05:33+02:00

RECORD THIS in the manuscript's data availability statement or
methods section, e.g.:
  "Phase 1 data was fetched from the dbt-labs/dbt-llm-sl-bench
  repository at commit c377468c59 (2026-04-07)."

Saved provenance record to ./phase1_data_provenance.txt
Tables found:
  challenge_text_info: 11 rows
  sql_answers: 5170 rows
  sql_answers_with_metadata: 5170 rows
Questions (challenge_text_info)      : 11  (manuscript states: 11)
Total records (sql_answers)          : 5170  (manuscript states: 5,170)
Distinct models                      : 24  (manuscript states: 24)

Records per method:
           method     n
0             sql  2585
1  semantic_layer  2585

✓ Question count and total record count MATCH the manuscript.

Distinct models in this data:
  anthropic:claude-opus-4-6:effort=h